The next word predictor

In [1]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [2]:
# Load data
data = """Hi John, I hope you are doing well.
It was great meeting you last week.
Please let me know if you need any help.
Looking forward to your response.
Thank you for your time and consideration.
I wanted to follow up regarding our last discussion.
Could you please share the updated report?
Let me know your availability for a quick meeting.
I appreciate your quick response.
Have a great day ahead!
Best regards, Jahangir"""

In [3]:
# --- Tokenization ---
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])
total_words = len(tokenizer.word_index) + 1

In [4]:
tokenizer.word_index

{'you': 1,
 'your': 2,
 'i': 3,
 'great': 4,
 'meeting': 5,
 'last': 6,
 'please': 7,
 'let': 8,
 'me': 9,
 'know': 10,
 'to': 11,
 'response': 12,
 'for': 13,
 'a': 14,
 'quick': 15,
 'hi': 16,
 'john': 17,
 'hope': 18,
 'are': 19,
 'doing': 20,
 'well': 21,
 'it': 22,
 'was': 23,
 'week': 24,
 'if': 25,
 'need': 26,
 'any': 27,
 'help': 28,
 'looking': 29,
 'forward': 30,
 'thank': 31,
 'time': 32,
 'and': 33,
 'consideration': 34,
 'wanted': 35,
 'follow': 36,
 'up': 37,
 'regarding': 38,
 'our': 39,
 'discussion': 40,
 'could': 41,
 'share': 42,
 'the': 43,
 'updated': 44,
 'report': 45,
 'availability': 46,
 'appreciate': 47,
 'have': 48,
 'day': 49,
 'ahead': 50,
 'best': 51,
 'regards': 52,
 'jahangir': 53}

In [5]:
# Convert text to sequences of words
input_sequences = []
for line in data.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

In [6]:
token_list

[51, 52, 53]

In [7]:
# Pad sequences to same length
from tensorflow.keras.preprocessing.sequence import pad_sequences
max_seq_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_seq_len, padding='post'))


In [8]:
# Split into predictors and label
X = input_sequences[:,:-1]
y = input_sequences[:,-1]
y = to_categorical(y, num_classes=total_words)

In [9]:
# --- Model ---
model = Sequential()
model.add(Embedding(total_words, 10, input_length=max_seq_len-1))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [13]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=100, verbose=1)

Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.9264 - loss: 0.1192
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9578 - loss: 0.0889
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9370 - loss: 0.1187
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9578 - loss: 0.0829
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9683 - loss: 0.0672
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9578 - loss: 0.0807
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9683 - loss: 0.0668
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9578 - loss: 0.0923
Epoch 9/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9683 - loss: 0.0830
Epoch 10/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.9578 - loss: 0.0945
Epoch 11/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9370 - loss: 0.1082
Epoch 12/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9683 - lo

In [15]:
def predict_next_word(model, tokenizer, text, max_seq_len):
    token_list = tokenizer.texts_to_sequences([text])[0]
    token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')
    predicted = model.predict(token_list, verbose=0)
    predicted_index = np.argmax(predicted, axis=1)[0]
    for word, index in tokenizer.word_index.items():
        if index == predicted_index:
            return word
    return None # Return None if the predicted index is not in the word index

seed_text = "Please let me"
next_word = predict_next_word(model, tokenizer, seed_text, max_seq_len)
if next_word:
    print(seed_text + " " + next_word)
else:
    print(f"Could not predict the next word for '{seed_text}'")

Could not predict the next word for 'Please let me'
